# 01 — EDA: NIH Malaria Cell Images

**UCS321 EST Project — Statement #4 (Biocon Ltd.)**

Explore the dataset before training: class balance, image-size distribution, sample grids, augmentation preview, addressing the *noise* and *variability* concerns from the problem statement.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

from src.config import CLASSES, DATA_RAW, FIGURES_DIR, IMG_SIZE, set_seeds
from src.data_loader import build_manifest, load_split_manifests, get_datasets

set_seeds()
sns.set_theme(style='whitegrid')
plt.rcParams['figure.facecolor'] = 'white'

## 1. Build the manifest and confirm class balance

In [ ]:
df = build_manifest()
print(f'Total images: {len(df):,}')
counts = df['label'].value_counts().sort_index()
counts.index = [CLASSES[i] for i in counts.index]
counts

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(counts.index, counts.values, color=['#34d399', '#ef4444'])
ax.set_title('Class balance — NIH Malaria Cell Images')
ax.set_ylabel('Count')
for i, v in enumerate(counts.values):
    ax.text(i, v + 50, f'{v:,}', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_class_balance.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Image size distribution (raw images vary in dimensions)

In [ ]:
sample_paths = df.sample(500, random_state=42)['path'].values
sizes = []
for p in sample_paths:
    with Image.open(p) as im:
        sizes.append(im.size)
sizes_df = pd.DataFrame(sizes, columns=['width', 'height'])
print(sizes_df.describe())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(sizes_df['width'], bins=30, color='#2dd4bf', edgecolor='white')
axes[0].set_title('Image width distribution (px)')
axes[0].axvline(IMG_SIZE, color='#fbbf24', linestyle='--', label=f'Resize target: {IMG_SIZE}')
axes[0].legend()
axes[1].hist(sizes_df['height'], bins=30, color='#2dd4bf', edgecolor='white')
axes[1].set_title('Image height distribution (px)')
axes[1].axvline(IMG_SIZE, color='#fbbf24', linestyle='--', label=f'Resize target: {IMG_SIZE}')
axes[1].legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_size_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Sample image grids — visualize variability and noise

In [ ]:
def grid(label_idx, title, n=8):
    paths = df[df['label'] == label_idx].sample(n, random_state=42)['path'].values
    fig, axes = plt.subplots(2, 4, figsize=(12, 6))
    for ax, p in zip(axes.flat, paths):
        ax.imshow(Image.open(p))
        ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    return fig

fig = grid(0, 'Uninfected RBCs')
fig.savefig(FIGURES_DIR / 'eda_samples_uninfected.png', dpi=150, bbox_inches='tight')
plt.show()
fig = grid(1, 'Parasitized RBCs')
fig.savefig(FIGURES_DIR / 'eda_samples_parasitized.png', dpi=150, bbox_inches='tight')
plt.show()

**Observations on noise/variability:** The raw images vary in stain darkness, lighting, slight focus differences, and orientation — consistent with the heterogeneity called out in the problem statement. The augmentation pipeline (flip / rotation / brightness / contrast / Gaussian noise) directly addresses this.

## 4. Splits

In [ ]:
splits = load_split_manifests()
rows = []
for name, sdf in splits.items():
    counts = sdf['label'].value_counts().sort_index()
    rows.append({
        'split': name, 'total': len(sdf),
        CLASSES[0]: counts.get(0, 0),
        CLASSES[1]: counts.get(1, 0)
    })
split_table = pd.DataFrame(rows)
split_table

## 5. Preview augmentation pipeline

In [ ]:
import tensorflow as tf
ds = get_datasets(batch_size=8)
batch_imgs, batch_labels = next(iter(ds['train']))
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, img, lbl in zip(axes.flat, batch_imgs, batch_labels):
    ax.imshow(img.numpy())
    ax.set_title(CLASSES[int(lbl)], fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle('Augmented training samples', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_augmented_samples.png', dpi=150, bbox_inches='tight')
plt.show()

**Summary**
- Dataset is balanced (50/50) → class-weighting not strictly needed but kept for safety.
- Raw image sizes vary; we resize to 128×128.
- Augmentation expands variability and injects noise — addresses the problem statement directly.
- 70/15/15 stratified split → train/val/test.